# Isolation Forest for Network Traffic Anomaly Detection

**Research question:** Can my Isolation Forest, trained on benign UNSW-NB15 network flows, identify attacks as unusual? How does it compare with scikit-learn?



## 1. Initial Setup


In [ ]:
import numpy as np
import pandas as pd
import random

from typing import List, Dict

from sklearn.ensemble import IsolationForest as SklearnIsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.metrics import confusion_matrix, precision_score, recall_score

In [ ]:
BASE_URL: str = (
    "https://raw.githubusercontent.com/"
    "lka1459/isolation-forest-from-scratch/main/data"
)
train_df: pd.DataFrame = pd.read_csv(f"{BASE_URL}/UNSW_NB15_training-set.csv")
test_df: pd.DataFrame = pd.read_csv(f"{BASE_URL}/UNSW_NB15_testing-set.csv")

Training rows: 175341 | Test rows: 82332


In [ ]:
# Experiment Parameters
SAMPLE_SIZE: int = 256
N_TREES: int = 10
MAX_DEPTH: int = 8

FEATURES: List[str] = [
    "dur", "spkts", "dpkts", "sbytes", "dbytes",
    "rate", "sload", "dload", "sinpkt", "dinpkt"
]

FEATURES_V2: List [str] = FEATURES + [
    "ct_dst_ltm",
    "ct_src_dport_ltm",
    "ct_dst_sport_ltm",
    "ct_srv_dst",
]

## 2. Isolation Tree



In [ ]:
class Isolation_Tree:
    def __init__(self, df: pd.DataFrame, max_depth: int):
        self.df: pd.DataFrame = df
        self.max_depth: int = max_depth
        self.root: dict = self.build_tree(self.df)

    def random_feature(self, node_data: pd.DataFrame) -> str:
        """
        Picks a random feature for the isolation tree.
        """
        features: List[str] = [feature for feature in node_data.columns if node_data[feature].nunique() > 1]
        rand_features: str = random.choice(features)

        return rand_features

    def split_values(self, feature: str, data: pd.DataFrame) -> float:
        """
        Splits the instances of a node randomly.
        """
        lowest_int: np.float64  = data[feature].min()
        highest_int: np.float64 = data[feature].max()
        rand_split: float = random.uniform(lowest_int, highest_int)

        return rand_split

    def partition_data(self, node_data: pd.DataFrame, select_feature: str, split_value: float):
        """
        Partitions the data of the selected parent node into two child nodes.
        """
        left_node: pd.DataFrame = node_data[node_data[select_feature] < split_value]
        right_node: pd.DataFrame = node_data[node_data[select_feature] >= split_value]

        return left_node, right_node

    def build_tree(self, node_data: pd.DataFrame, depth: int = 0) -> Dict[str, str]:
        """
        Build a single isolation tree based on a randomly selected feature.
        """
        if (
            depth >= self.max_depth
            or len(node_data) <= 1
            or all(node_data[col].nunique() <= 1 for col in node_data.columns)
            ):
            return {"leaf_size": len(node_data)}

        rand_feature: str = self.random_feature(node_data)
        rand_split: float = self.split_values(rand_feature, node_data)
        left_node, right_node = self.partition_data(node_data, rand_feature, rand_split)

        return {
        "feature": rand_feature,
        "split_value": rand_split,
        "left_node": self.build_tree(left_node, depth + 1),
        "right_node": self.build_tree(right_node, depth + 1),
                }

    def return_path_length(self, row: pd.Series) -> float:
        """
        Returns path length of a row.
        """
        node: Dict[str, str] = self.root
        path_length: int = 0

        while "leaf_size" not in node:
            feature: str = node["feature"]
            split_value: float = node["split_value"]

            if row[feature] < split_value:
                node = node["left_node"]
            else:
                node = node["right_node"]

            path_length += 1

        leaf_size: int = node["leaf_size"]

        return path_length + self.expected_extra_length(leaf_size)

    def expected_extra_length(self, n: int) -> float:
        """
        Calculates the extra length of a path length based on leaf nodes size.
        """
        if n <= 1:
            return 0

        if n == 2:
            return 1

        harmonic: float = sum(1 / i for i in range(1, n))
        expected_length: float = (2 * harmonic) - (2 * (n - 1)/ n)

        return expected_length

## 3. Custom Isolation Forest


In [ ]:
class Isolation_Forest:
    def __init__(self, df: pd.DataFrame, sample_size: int, n_estimators: int, max_depth: int):
        self.df: pd.DataFrame = df
        self.sample_size: int = sample_size
        self.n_estimators: int = n_estimators
        self.trees: List[Isolation_Tree] = self.build_ensemble(df, max_depth)

    def random_sample(self, data_set: pd.DataFrame) -> List[pd.DataFrame]:
        """
        Returns a random sample of a dataset for isolation.
        """
        sample_size: int = self.sample_size
        sample_list: List[pd.DataFrame] = []

        for _ in range(0, self.n_estimators):
            samples: List[pd.Series] = []
            df_shuffled: pd.DataFrame = data_set.sample(frac=1,
                                                        random_state=42 + len(sample_list)
                                                        ).reset_index(drop=True)
            for j in range(0, sample_size):
                samples.append(df_shuffled.iloc[j])

            df: pd.DataFrame = pd.DataFrame(samples)
            sample_list.append(df)

        return sample_list

    def build_ensemble(self, data_set: pd.DataFrame, max_depth: int) -> List[Isolation_Tree]:
        """
        Builds an Isolation Tree for each sampled dataset.
        """
        sample_list: List[pd.DataFrame] = self.random_sample(data_set)
        tree_list: List[Isolation_Tree] = []

        for sample in sample_list:
            isolation_tree: Isolation_Tree = Isolation_Tree(sample, max_depth)
            tree_list.append(isolation_tree)

        return tree_list

    def average_path_length(self, row: pd.Series) -> float:
        """
        Return the average path length of the Isolation Tree forest.
        """
        lengths: List[float] = []

        for tree in self.trees:
            lengths.append(tree.return_path_length(row))

        average: float = sum(lengths) / len(lengths)

        return average

    def anomaly_score(self, row: pd.Series) -> float:
        """
        Converts average into an anomaly score.
        """
        c_sample: float = self.trees[0].expected_extra_length(self.sample_size)
        average: float = self.average_path_length(row)

        ratio: float = 2 ** (-average / c_sample)

        return ratio

## 4. Data split and baseline feature set



In [ ]:
fit_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df["label"],
)

X_fit: pd.DataFrame = fit_df.loc[fit_df["label"] == 0, FEATURES].sample(
    n=2_000, random_state=42
)

validation: pd.DataFrame = pd.concat([
    val_df.loc[val_df["label"] == 0].sample(n=500, random_state=42),
    val_df.loc[val_df["label"] == 1].sample(n=500, random_state=42),
]).sample(frac=1, random_state=42)

X_val: pd.DataFrame = validation[FEATURES]
y_val: pd.Series = validation["label"]

### 4.1 Baseline comparison



In [ ]:
random.seed(42)

my_forest: Isolation_Forest = Isolation_Forest(
    X_fit,
    sample_size=SAMPLE_SIZE,
    n_estimators=N_TREES,
    max_depth=MAX_DEPTH,
)

my_scores: pd.Series = X_val.apply(my_forest.anomaly_score, axis=1)

In [ ]:
sklearn_forest: SklearnIsolationForest = SklearnIsolationForest(
    n_estimators=N_TREES,
    max_samples=SAMPLE_SIZE,
    contamination="auto",
    random_state=42,
)
sklearn_forest.fit(X_fit)

sklearn_scores: pd.Series = pd.Series(
    -sklearn_forest.score_samples(X_val),
    index=X_val.index,
)

In [ ]:
baseline_results = pd.DataFrame({
    "label": y_val,
    "my_score": my_scores,
    "sklearn_score": sklearn_scores,
})

def ranking_table(labels, custom_scores, reference_scores):
    return pd.DataFrame({
        "ROC-AUC": [roc_auc_score(labels, custom_scores),
                    roc_auc_score(labels, reference_scores)],
        "Average precision": [average_precision_score(labels, custom_scores),
                              average_precision_score(labels, reference_scores)],
    }, index=["Custom forest", "scikit-learn"]).round(3)

print("Validation (10 features; 500 benign and 500 attack flows)")
display(ranking_table(y_val, my_scores, sklearn_scores))

Validation (10 features; 500 benign and 500 attack flows)


,ROC-AUC,Average precision
Custom forest,0.601,0.547
scikit-learn,0.657,0.632


### 4.2 Where does the baseline struggle?


In [ ]:
baseline_results["attack_cat"] = validation["attack_cat"]
category_summary = (
    baseline_results.groupby("attack_cat")
    .agg(count=("label", "size"), custom_mean=("my_score", "mean"),
         sklearn_mean=("sklearn_score", "mean"))
    .sort_values("custom_mean", ascending=False)
)
display(category_summary.round(3))

,count,custom_mean,sklearn_mean
attack_cat,,,
Shellcode,4,0.408,0.450
DoS,52,0.402,0.424
Exploits,127,0.401,0.434
Generic,173,0.399,0.422
Fuzzers,77,0.398,0.416
Normal,500,0.389,0.390
Backdoor,4,0.389,0.393
Analysis,16,0.380,0.389
Reconnaissance,47,0.358,0.373


## 5. Feature experiment



In [ ]:
# Train on benign flows only.
X_fit2: pd.DataFrame = fit_df.loc[fit_df["label"] == 0, FEATURES_V2].sample(
    n=2_000, random_state=42
)

X_val2: pd.DataFrame = validation[FEATURES_V2]

In [ ]:
random.seed(42)

my_forest: Isolation_Forest = Isolation_Forest(
    X_fit2,
    sample_size=SAMPLE_SIZE,
    n_estimators=N_TREES,
    max_depth=MAX_DEPTH,
)

my_scores: pd.Series = X_val2.apply(my_forest.anomaly_score, axis=1)

results: pd.DataFrame = pd.DataFrame({
    "label": y_val,
    "anomaly_score": my_scores,
})

In [ ]:
sklearn_forest: SklearnIsolationForest = SklearnIsolationForest(
    n_estimators=N_TREES,
    max_samples=SAMPLE_SIZE,
    contamination="auto",
    random_state=42,
)
sklearn_forest.fit(X_fit2)

sklearn_scores: pd.Series = pd.Series(
    -sklearn_forest.score_samples(X_val2),
    index=X_val.index,
)

In [ ]:
results = pd.DataFrame({
    "label": y_val,
    "my_score": my_scores,
    "sklearn_score": sklearn_scores,
})

print("Validation (14 features; 500 benign and 500 attack flows)")
display(ranking_table(y_val, my_scores, sklearn_scores))
print("Mean anomaly scores by true label (higher = more unusual)")
display(results.groupby("label")[["my_score", "sklearn_score"]].mean().round(3))

Validation (14 features; 500 benign and 500 attack flows)


,ROC-AUC,Average precision
Custom forest,0.735,0.771
scikit-learn,0.707,0.779


Mean anomaly scores by true label (higher = more unusual)


,my_score,sklearn_score
label,,
0,0.372,0.393
1,0.462,0.490


## 6. Alert threshold and held-out evaluation

In [ ]:
my_threshold: float = float(
    results.loc[results["label"] == 0, "my_score"].quantile(0.95)
)
sklearn_threshold: float = float(
    results.loc[results["label"] == 0, "sklearn_score"].quantile(0.95)
)

validation_alerts: list[dict[str, str | float | int]] = []

name: str
score_column: str
threshold: float
for name, score_column, threshold in [
    ("Custom forest", "my_score", my_threshold),
    ("scikit-learn", "sklearn_score", sklearn_threshold),
]:
    alerts: pd.Series = (results[score_column] >= threshold).astype(int)

    counts: tuple[int, int, int, int] = tuple(
        int(value)
        for value in confusion_matrix(y_val, alerts, labels=[0, 1]).ravel()
    )
    tn, fp, fn, tp = counts

    validation_alerts.append({
        "Model": name,
        "Threshold": threshold,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Precision": float(precision_score(y_val, alerts)),
        "Recall": float(recall_score(y_val, alerts)),
    })

validation_summary: pd.DataFrame = (
    pd.DataFrame(validation_alerts).set_index("Model")
)
display(validation_summary.round(3))

,Threshold,TN,FP,FN,TP,Precision,Recall
Model,,,,,,,
Custom forest,0.500,475,25,309,191,0.884,0.382
scikit-learn,0.527,475,25,280,220,0.898,0.440


In [ ]:
X_test: pd.DataFrame = test_df[FEATURES_V2]
y_test: pd.Series = test_df["label"]

my_test_scores: pd.Series = X_test.apply(my_forest.anomaly_score, axis=1)
sklearn_test_scores: np.ndarray = -sklearn_forest.score_samples(X_test)

test_summary: pd.DataFrame = ranking_table(
    y_test, my_test_scores, sklearn_test_scores
)
test_alerts: list[dict[str, str | float | int]] = []

name: str
scores: pd.Series | np.ndarray
threshold: float
for name, scores, threshold in [
    ("Custom forest", my_test_scores, my_threshold),
    ("scikit-learn", sklearn_test_scores, sklearn_threshold),
]:
    alerts: pd.Series | np.ndarray = (scores >= threshold).astype(int)

    counts: tuple[int, int, int, int] = tuple(
        int(value)
        for value in confusion_matrix(y_test, alerts, labels=[0, 1]).ravel()
    )
    tn, fp, fn, tp = counts

    test_alerts.append({
        "Model": name,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Precision": float(precision_score(y_test, alerts, zero_division=0)),
        "Recall": float(recall_score(y_test, alerts, zero_division=0)),
    })

test_summary = test_summary.join(
    pd.DataFrame(test_alerts).set_index("Model")
)

print(
    f"Held-out test: {len(X_test):,} flows "
    f"({sum(y_test == 0):,} benign; {sum(y_test == 1):,} attacks)"
)
print("TN = correct benign, FP = false alert, FN = missed attack, TP = detected attack")
display(test_summary.round(3))

Held-out test: 82,332 flows (37,000 benign; 45,332 attacks)
TN = correct benign, FP = false alert, FN = missed attack, TP = detected attack


,ROC-AUC,Average precision,TN,FP,FN,TP,Precision,Recall
Custom forest,0.735,0.789,34519,2481,31511,13821,0.848,0.305
scikit-learn,0.724,0.819,35283,1717,23219,22113,0.928,0.488
